# CLIP-ReIdent: basketball disappearance / return
Run the released Paper ViT-L/14 checkpoint on saved tracker observations.
Select a Colab GPU. Upload the refreshed private source archive using
`scripts/prepare_colab.ps1`. No source video is uploaded by this notebook.
The model and outputs are saved to Drive. First download: ~3.8 GB.
Identity links are experimental hypotheses; scores are not probabilities.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, tempfile, zipfile, os
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TacticalAI')
# Set this to the folder containing attack_clip/tracktrack and attack_clip2/tracktrack.
SOURCE_RUN = ROOT / 'outputs/occlusion-20261015T204150Z-029a7a35/reid'
ASSETS = ROOT / 'models/clip-reident'
OUTPUT = ROOT / 'outputs' / ('clip-reident-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
REPO = Path(tempfile.mkdtemp(prefix='tacticalai-reid-'))
with zipfile.ZipFile(ROOT / 'code/tacticalai-source.zip') as archive:
    archive.extractall(REPO)
os.chdir(REPO)
assert (REPO / 'src/tacticalai/reidentify_cli.py').is_file(), 'Refresh the source archive first'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[clip-reident,geometry,preview]'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'scripts/prepare_clip_reident.py', '--root', str(ASSETS)], check=True)


The following settings reproduce the second development trial, not a validated winner.
The saved source run retains its original buffer (30 in the historical run).
New tracking configs use 45. The offline maximum gap is independent.

The final gallery-review stage uses full-video crop selection and one-to-one matching with margin 0.001, calibrated on seven development pairs. It is not held-out validation.


In [ ]:
CHECKPOINT = ASSETS / 'model/ViT-L-14_openai/Paper/weights_e4.pth'
for clip in ('attack_clip', 'attack_clip2'):
    subprocess.run([sys.executable, '-m', 'tacticalai.reidentify_cli',
        '--video', str(ROOT / 'data/raw' / f'{clip}.mp4'),
        '--run', str(SOURCE_RUN / clip / 'tracktrack'),
        '--output', str(OUTPUT / clip), '--checkpoint', str(CHECKPOINT),
        '--device', 'cuda:0', '--batch-size', '4',
        '--threshold', '0.98', '--margin', '0.003'], check=True)
ENRICHED = OUTPUT / 'full-video-galleries'
subprocess.run([sys.executable, '-m', 'tacticalai.enrich_galleries_cli', '--cache-root', str(OUTPUT), '--output', str(ENRICHED), '--device', 'cuda:0'], check=True)
REVIEW = OUTPUT / 'gallery-review'
subprocess.run([sys.executable, '-m', 'tacticalai.gallery_reid_cli', '--cache-root', str(ENRICHED), '--output', str(REVIEW), '--comparison', 'one_to_one', '--min-spacing', '5', '--margin', '0.001'], check=True)
print('Persistent whole-ID review:', REVIEW)


Serve the gallery-review folder locally and open index.html for whole-ID counts, synchronized views and gallery evidence.
`frames.jsonl` preserves raw track IDs and adds provisional player IDs.
Compare true pre/post-occlusion identities manually before reporting recovery accuracy.


## Optional temporal identity consistency experiment
Reuse the exact source tracks; extract denser time-indexed appearance features on GPU.
This stage separates persistent appearance changes only when tracking also supplies a nearby break or abrupt jump.
Unclear transitions stay unknown. Original IDs are preserved alongside separate person hypotheses.
These are development heuristics; no held-out identity accuracy is claimed.


In [ ]:
TEMPORAL_FEATURES = OUTPUT / 'temporal-features'
TEMPORAL_REVIEW = OUTPUT / 'temporal-review'
subprocess.run([sys.executable, '-m', 'tacticalai.temporal_features_cli',
    '--original', str(OUTPUT), '--enriched', str(ENRICHED),
    '--output', str(TEMPORAL_FEATURES), '--device', 'cuda:0'], check=True)
subprocess.run([sys.executable, '-m', 'tacticalai.temporal_reid_cli',
    '--cache-root', str(TEMPORAL_FEATURES), '--output', str(TEMPORAL_REVIEW)], check=True)
print('Download the temporal-review folder and serve it locally:', TEMPORAL_REVIEW)
